In [ ]:
from pyspark.sql import SparkSession

# Create top_merchants spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 12:10:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
import pandas as pd 

top_merchants = pd.read_csv("top_100_merchants.csv")
merch_ranking = pd.read_csv("merchant_segment_rankings_all.csv")   # note: "rankings", with the g

In [7]:
import numpy as np

merch_ranking = merch_ranking.sort_values('overall_rank').reset_index(drop=True)
top100_from_all = merch_ranking.head(100)

print('abn unique:', merch_ranking.merchant_abn.is_unique)                                      # True
print('ranks contiguous:', (merch_ranking.overall_rank.values == np.arange(1, len(merch_ranking)+1)).all())  # True
print('score sorted desc:', merch_ranking.ranking_score.is_monotonic_decreasing)              # True
print('top file == head(100):', set(top_merchants.merchant_abn) == set(top100_from_all.merchant_abn))  # True
print('value = avg*count, max gap:',
      (merch_ranking.avg_transaction_value * merch_ranking.transaction_count
       - merch_ranking.total_transaction_value).abs().max())                                   # 0.0
print('customers > txns:', (merch_ranking.unique_customers > merch_ranking.transaction_count).sum())  # 0
print('fraud_flag_observed_count == transaction_count:',
      (merch_ranking.fraud_flag_observed_count == merch_ranking.transaction_count).mean())    # 1.0  <- odd, ask the friend what this column is
print(merch_ranking.first_transaction.min(), merch_ranking.last_transaction.max())            # 2021-02-28 2022-10-26
nulls = merch_ranking.isna().sum()
print(nulls[nulls > 0].to_dict())   # industry/tier/take_rate 547, name/tags 396, growth 310

abn unique: True
ranks contiguous: True
score sorted desc: True
top file == head(100): True
value = avg*count, max gap: 1.862645149230957e-09
customers > txns: 0
fraud_flag_observed_count == transaction_count: 1.0
2021-02-28 2022-10-26
{'std_transaction_value': 14, 'recent_month_growth': 310, 'name_merchant': 396, 'tags': 396, 'customer_mean_disadvantage_score': 2, 'customer_mean_economic_resources_score': 2, 'predicted_3m_revenue': 37, 'industry': 547, 'merchant_tier': 547, 'take_rate': 547}


In [11]:
import numpy as np, pandas as pd

top_merchants = merch_ranking.sort_values('overall_rank').reset_index(drop=True)
top_merchants['in_top100'] = top_merchants.overall_rank <= 100
top_merchants['value_share'] = top_merchants.total_transaction_value / top_merchants.total_transaction_value.sum()

In [ ]:
print(top_merchants.shape)                   
missing = top_merchants.isna().mean()
print(missing[missing > 0].round(3))              
print([c for c in top_merchants.columns if top_merchants[c].nunique() == 1]) 

(4422, 66)
std_transaction_value                     0.003
recent_month_growth                       0.070
name_merchant                             0.090
tags                                      0.090
customer_mean_disadvantage_score          0.000
customer_mean_economic_resources_score    0.000
predicted_3m_revenue                      0.008
industry                                  0.124
merchant_tier                             0.124
take_rate                                 0.124
dtype: float64
['fraud_flag_coverage', 'fraud_data_review_flag']


In [ ]:
# Produces a percentage of how much of the transaction value is held by best ranked merchants k 
# k=10: checks the top 10 ranked merchants 
# k=100: the top 100 ranked merchants relevant to our analysis 
# k=442: top 442 ranked merchants which is also the top 10% of our ranked merchants since we have 4222 merchants in total. 
# k=1000: the top 1000 ranked merchants 
for k in (10, 100, 442, 1000):
    print(k, round(top_merchants.value_share.head(k).sum(), 3)) 

# Produces the percentage of how much of the transaction value is held by the top 100 merchants with purely transaction value (no ranking involved)
print('top 100 with trasaction value only (no ranking)', round(top_merchants.value_share.nlargest(100).sum(), 3)) 

10 0.035
100 0.27
442 0.508
1000 0.734
top 100 with trasaction value onlt (no ranking) 0.319


In [ ]:
top = top_merchants[top_merchants.in_top100]
rest = top_merchants[~top_merchants.in_top100]
similar = rest[rest.unique_customers >= top.unique_customers.min()]    # others as big as the top 100 (186 of them)

features = ['unique_customers', 'repeat_customer_share', 'avg_transaction_value', 'total_transaction_value',
            'monthly_value_cv', 'unique_customer_sa2s', 'take_rate']
print(pd.DataFrame({'top 100': top[features].median(),
                    'all others': rest[features].median(),
                    'others of similar size': similar[features].median()}).round(3))

                             top 100  all others  others of similar size
unique_customers           18228.500     350.000               11493.000
repeat_customer_share          0.542       0.008                   0.290
avg_transaction_value        150.622     355.568                  57.957
total_transaction_value  7148015.778  132952.089             1004346.887
monthly_value_cv               0.283       0.401                   0.283
unique_customer_sa2s        1487.000     245.000                1464.500
take_rate                      4.450       4.620                   4.845


In [35]:
# Produces the count of unique merchants, the total value share, the total number of merchants in the top 100, 
# the average transaction value and the averafe take rate for each of the 7 merchant segments 
by_segment = top_merchants.groupby('segment').agg(merchants=('merchant_abn', 'count'), value_share=('value_share', 'sum'),
                                      top100=('in_top100', 'sum'), average_purchases=('avg_transaction_value', 'median'),
                                      take_rate=('take_rate', 'mean')).round(3)
print(by_segment.sort_values('value_share', ascending=False))

# Similarly produces total number of merchants, and the mean take rate for each of the merchant tiers of a, b, c
print(top_merchants.groupby('merchant_tier').agg(merchants=('merchant_abn', 'count'), take_rate=('take_rate', 'mean')).round(2))

                        merchants  value_share  top100  average_purchases  \
segment                                                                     
Electronics & Digital        1135        0.243      26            203.641   
Fashion & Lifestyle          1083        0.232      28            218.496   
Arts, Media & Services        696        0.148      13            430.433   
Automotive & Outdoor          478        0.138      14            512.196   
Unknown                       547        0.112       7           1273.067   
Other                         332        0.081       8            562.591   
Home & Living                 151        0.045       4            320.145   

                        take_rate  
segment                            
Electronics & Digital       4.598  
Fashion & Lifestyle         4.530  
Arts, Media & Services      4.454  
Automotive & Outdoor        4.591  
Unknown                       NaN  
Other                       4.407  
Home & Living     

In [ ]:
# Outputs the top 3 merchants with greater recent monthly growth
print(merch_ranking.nlargest(3, 'recent_month_growth')[['merchant_abn', 'name_merchant', 'segment', 'overall_rank',
                                                  'previous_month_value', 'recent_month_value', 'recent_month_growth']].round(1))
# Outputs the number of  merchant with only one customer 
print((merch_ranking.unique_customers == 1).sum())

# Outputs the number of merchant with only one customer but that customer makes up for half of their value 
print((merch_ranking.largest_customer_value_share > 0.5).sum())

# Outputs the 
print(merch_ranking.avg_transaction_value.quantile([.5, .9, .99]).round(0).to_dict())  

      merchant_abn                    name_merchant                 segment  \
2693   16496390781                              NaN                 Unknown   
4241   68889885215           Faucibus Lectus A Inc.  Arts, Media & Services   
4181   20415536115  Nibh Phasellus Nulla Industries           Home & Living   

      overall_rank  previous_month_value  recent_month_value  \
2693          2694                  30.4              8655.6   
4241          4242                  26.1               971.6   
4181          4182                 109.3              3878.4   

      recent_month_growth  
2693                283.7  
4241                 36.2  
4181                 34.5  
    merchant_abn              name_merchant                 segment  \
0    86868464441                        NaN                 Unknown   
1    47047735645   Aenean Gravida Institute  Arts, Media & Services   
2    39150153670       Aliquam Eu Institute  Arts, Media & Services   
3    29068447069              

In [ ]:
# Identifying the number of flagged merchants and how many times they have been flagged
flagged = merch_ranking[merch_ranking.any_flagged_transaction_count > 0]
print(len(flagged), flagged.any_flagged_transaction_count.value_counts().to_dict()) 

# Identifying the merchants with the high risk flag 
print(merch_ranking[merch_ranking.fraud_risk_flag][['overall_rank', 'transaction_count', 'fraud_rate_used']].round(3))

62 {1: 54, 2: 4, 3: 4}
      overall_rank  transaction_count  fraud_rate_used
2292          2293                 44            0.068
2982          2983                 11            0.091
3043          3044                 14            0.071
3065          3066                 16            0.062
3089          3090                  6            0.167
3363          3364                  3            0.333
4287          4288                  1            1.000
4295          4296                  1            1.000
-0.9959843828559267


In [36]:
big_base        = merch_ranking.unique_customers >= merch_ranking.unique_customers.quantile(.75)
small_base      = merch_ranking.unique_customers <= merch_ranking.unique_customers.quantile(.25)
large_purchases = merch_ranking.avg_transaction_value >= merch_ranking.avg_transaction_value.quantile(.75)

groups = {
    'Small customer base, large purchases': small_base & large_purchases,
    'Large customer base, some fraud flag': big_base & (merch_ranking.any_flagged_transaction_count > 0),
    'Low history':                          merch_ranking.low_history_flag,
}

rows = []
for name, mask in groups.items():
    g = merch_ranking[mask]
    rows.append({'group': name,
                 'merchants': len(g),
                 'in top 100': int(g.in_top100.sum()),
                 'median rank': g.overall_rank.median(),
                 'share of value (%)': round(g.value_share.sum() * 100, 1)})

summary = pd.DataFrame(rows).set_index('group')
print(summary)                                     

AttributeError: 'DataFrame' object has no attribute 'in_top100'